# Checkpoint 9 — raw subscribers, refined tiers, and clean average

This controlled experiment combines raw subscriber count, the seven refined subscriber tiers, and leakage-safe historical average channel views per video. Raw cumulative channel views remain excluded.

The horizon rows, channel-grouped folds, target transformation, XGBoost parameters, and untouched reserved test remain identical.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.train_checkpoint5_models import (
    HORIZONS,
    MODEL_NAME,
    load_horizon_checkpoint,
    train_all_horizons,
)
from viewcastlk_ml.horizon_preprocessing import HorizonDatasetPreprocessor

ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'checkpoint9_all_channel_scale'
FEATURES = (
    'ch_subs_at_publish',
    'subscriber_tier',
    'ch_avg_views_per_video_at_publish',
)

loaded = {}
inventory_rows = []
for horizon in HORIZONS:
    X, y, assignments, _, _ = load_horizon_checkpoint(PROJECT_ROOT, horizon)
    loaded[horizon] = (X, y, assignments)
    assert set(FEATURES).issubset(X.columns)
    assert 'ch_views_at_publish' not in X.columns
    inventory_rows.append({
        'horizon_days': horizon,
        'rows': len(X),
        'raw_subscriber_rows': int(X['ch_subs_at_publish'].notna().sum()),
        'subscriber_tiers': int(X['subscriber_tier'].nunique()),
        'clean_average_rows': int(X['ch_avg_views_per_video_at_publish'].notna().sum()),
        'clean_average_percent': 100 * X['ch_avg_views_per_video_at_publish'].notna().mean(),
    })

display(pd.DataFrame(inventory_rows).round(2))

   horizon_days   rows  ...  clean_average_rows  clean_average_percent
0             7  20663  ...               13828                  66.92
1            14  15685  ...                8073                  51.47
2            21  15100  ...                2570                  17.02
3            30  14753  ...                   0                   0.00

[4 rows x 6 columns]


In [2]:
# Show actual numeric model inputs from a validation fold.
X7, y7, assignments7 = loaded[7]
training_mask = assignments7['partition'].eq('development') & ~assignments7['cv_validation_fold'].eq(1)
validation_mask = assignments7['partition'].eq('development') & assignments7['cv_validation_fold'].eq(1)
training_positions = assignments7.loc[training_mask, 'horizon_row_position'].astype(int).to_numpy()
validation_positions = assignments7.loc[validation_mask, 'horizon_row_position'].astype(int).to_numpy()

preprocessor = HorizonDatasetPreprocessor()
preprocessor.fit(X7.iloc[training_positions], np.log1p(y7.iloc[training_positions]))
source_preview = X7.iloc[validation_positions[:8]]
model_preview = preprocessor.transform(source_preview)
tier_features = [column for column in model_preview if column.startswith('subscriber_tier_')]

display(pd.concat([
    source_preview[list(FEATURES)].reset_index(drop=True),
    model_preview[['ch_subs_at_publish', 'ch_avg_views_per_video_at_publish'] + tier_features].reset_index(drop=True),
], axis=1))

assert 'ch_subs_at_publish' in model_preview.columns
assert 'ch_avg_views_per_video_at_publish' in model_preview.columns
assert len(tier_features) == 7
assert 'ch_views_at_publish' not in model_preview.columns
assert not np.isinf(model_preview.to_numpy(dtype=float)).any()
print('PASS: all three intended channel-scale representations are present; raw cumulative views are absent.')

   ch_subs_at_publish  ... subscriber_tier_under_1k
0           3770000.0  ...                      0.0
1           3770000.0  ...                      0.0
2           3780000.0  ...                      0.0
3              7340.0  ...                      0.0
4           3770000.0  ...                      0.0
5           3770000.0  ...                      0.0
6            589000.0  ...                      0.0
7           3780000.0  ...                      0.0

[8 rows x 12 columns]
PASS: all three intended channel-scale representations are present; raw cumulative views are absent.


In [3]:
training_run = train_all_horizons(
    project_root=PROJECT_ROOT,
    output_dir=ARTIFACT_DIR,
    n_estimators=800,
    n_jobs=4,
    include_llm_scores=False,
)

display(training_run['summary'][[
    'horizon_days', 'model', 'rows', 'mape_nonzero_pct',
    'median_ape_nonzero_pct', 'smape_pct', 'rmsle', 'log_r2'
]])


Training independent day-7 model
day 7 fold 1/5: RMSLE=1.9648, median APE=155.65%, best trees=28
day 7 fold 2/5: RMSLE=1.9768, median APE=87.50%, best trees=170
day 7 fold 3/5: RMSLE=2.1196, median APE=94.74%, best trees=60
day 7 fold 4/5: RMSLE=1.9814, median APE=83.76%, best trees=35
day 7 fold 5/5: RMSLE=2.1578, median APE=98.76%, best trees=39

Training independent day-14 model
day 14 fold 1/5: RMSLE=1.9884, median APE=188.67%, best trees=17
day 14 fold 2/5: RMSLE=1.9531, median APE=83.84%, best trees=25
day 14 fold 3/5: RMSLE=2.1128, median APE=94.42%, best trees=431
day 14 fold 4/5: RMSLE=1.8670, median APE=89.17%, best trees=150
day 14 fold 5/5: RMSLE=1.8771, median APE=88.04%, best trees=106

Training independent day-21 model
day 21 fold 1/5: RMSLE=1.9264, median APE=99.82%, best trees=145
day 21 fold 2/5: RMSLE=1.9058, median APE=86.03%, best trees=50
day 21 fold 3/5: RMSLE=1.9740, median APE=87.14%, best trees=61
day 21 fold 4/5: RMSLE=1.9454, median APE=89.07%, best trees=4

In [4]:
# Compare every subscriber representation tested on the same folds.
run_directories = {
    'raw_only': PROJECT_ROOT / 'artifacts' / 'checkpoint5_xgboost',
    'raw_plus_coarse_tier': PROJECT_ROOT / 'artifacts' / 'checkpoint6_subscriber_tier',
    'refined_tier_only': PROJECT_ROOT / 'artifacts' / 'checkpoint7_refined_tier_only',
    'refined_tier_plus_clean_average': PROJECT_ROOT / 'artifacts' / 'checkpoint8_clean_channel_average',
    'raw_plus_refined_tier_plus_clean_average': ARTIFACT_DIR,
}
comparison_rows = []
for run_name, directory in run_directories.items():
    summary = pd.read_csv(directory / 'cv_summary_metrics.csv', dtype={'horizon_days': str})
    for horizon in [str(h) for h in HORIZONS] + ['combined']:
        result = summary[
            summary['horizon_days'].eq(horizon)
            & summary['model'].eq(MODEL_NAME)
        ].iloc[0]
        comparison_rows.append({
            'horizon_days': horizon,
            'configuration': run_name,
            'rmsle': result['rmsle'],
            'mape_pct': result['mape_nonzero_pct'],
            'median_ape_pct': result['median_ape_nonzero_pct'],
            'smape_pct': result['smape_pct'],
            'log_r2': result['log_r2'],
        })

comparison = pd.DataFrame(comparison_rows)
display(comparison.round(4))
print('Combined comparison')
display(comparison[comparison['horizon_days'].eq('combined')].sort_values('rmsle').round(4))

importance = pd.read_csv(ARTIFACT_DIR / 'feature_importance_gain.csv')
important_channel_features = importance[
    importance['feature'].isin(['ch_subs_at_publish', 'ch_avg_views_per_video_at_publish'])
    | importance['feature'].str.startswith('subscriber_tier_')
].sort_values(['horizon_days', 'gain_share_within_horizon'], ascending=[True, False])
display(important_channel_features[['horizon_days', 'feature', 'gain_share_within_horizon']].round(5))

   horizon_days                             configuration  ...  smape_pct  log_r2
0             7                                  raw_only  ...   117.0905  0.3005
1            14                                  raw_only  ...   117.1304  0.2991
2            21                                  raw_only  ...   114.0420  0.3665
3            30                                  raw_only  ...   114.5122  0.3753
4      combined                                  raw_only  ...   115.8307  0.3329
5             7                      raw_plus_coarse_tier  ...   116.3905  0.3352
6            14                      raw_plus_coarse_tier  ...   117.6134  0.3043
7            21                      raw_plus_coarse_tier  ...   114.8247  0.3564
8            30                      raw_plus_coarse_tier  ...   114.3245  0.3825
9      combined                      raw_plus_coarse_tier  ...   115.8630  0.3439
10            7                         refined_tier_only  ...   117.5701  0.2470
11           14 

In [5]:
# Artifact and leakage tests.
manifest = json.loads((ARTIFACT_DIR / 'training_manifest.json').read_text(encoding='utf-8'))
predictions = pd.read_csv(ARTIFACT_DIR / 'cv_predictions.csv')
reference_predictions = pd.read_csv(PROJECT_ROOT / 'artifacts' / 'checkpoint8_clean_channel_average' / 'cv_predictions.csv')
test_rows = []

def check(name, condition, detail=''):
    test_rows.append({'test': name, 'status': 'PASS' if bool(condition) else 'FAIL', 'detail': detail})

check('reserved test remains unevaluated', manifest['status'] == 'candidate_reserved_test_not_evaluated')
check('same OOF rows as previous run', set(zip(predictions['horizon_days'], predictions['horizon_row_position'])) == set(zip(reference_predictions['horizon_days'], reference_predictions['horizon_row_position'])))
check('predictions finite', np.isfinite(predictions.filter(like='predicted_').to_numpy(dtype=float)).all())

for record in manifest['models']:
    horizon = record['horizon_days']
    bundle = joblib.load(ARTIFACT_DIR / record['model_path'])
    X, y, assignments = loaded[horizon]
    development_positions = set(assignments.loc[assignments['partition'].eq('development'), 'horizon_row_position'].astype(int))
    reserved_positions = set(assignments.loc[assignments['partition'].eq('test_reserved'), 'horizon_row_position'].astype(int))
    predicted_positions = set(predictions.loc[predictions['horizon_days'].eq(horizon), 'horizon_row_position'].astype(int))
    tier_features = [feature for feature in bundle.feature_names if feature.startswith('subscriber_tier_')]
    sample_prediction = bundle.predict_views(X.iloc[[min(development_positions)]])

    check(f'day {horizon} raw subscribers saved', 'ch_subs_at_publish' in bundle.feature_names)
    check(f'day {horizon} clean average saved', 'ch_avg_views_per_video_at_publish' in bundle.feature_names)
    check(f'day {horizon} seven refined tiers saved', len(tier_features) == 7)
    check(f'day {horizon} raw cumulative views absent', 'ch_views_at_publish' not in bundle.feature_names)
    check(f'day {horizon} development coverage exact', predicted_positions == development_positions)
    check(f'day {horizon} reserved rows absent', predicted_positions.isdisjoint(reserved_positions))
    check(f'day {horizon} bundle reloads and predicts', len(sample_prediction) == 1 and np.isfinite(sample_prediction).all())

tests = pd.DataFrame(test_rows)
display(tests)
failures = tests[tests['status'].eq('FAIL')]
assert failures.empty, failures.to_string(index=False)
print(f'PASS: all {len(tests)} combined channel-scale checks succeeded.')

                                  test status detail
0    reserved test remains unevaluated   PASS       
1        same OOF rows as previous run   PASS       
2                   predictions finite   PASS       
3          day 7 raw subscribers saved   PASS       
4            day 7 clean average saved   PASS       
5      day 7 seven refined tiers saved   PASS       
6    day 7 raw cumulative views absent   PASS       
7     day 7 development coverage exact   PASS       
8           day 7 reserved rows absent   PASS       
9    day 7 bundle reloads and predicts   PASS       
10        day 14 raw subscribers saved   PASS       
11          day 14 clean average saved   PASS       
12    day 14 seven refined tiers saved   PASS       
13  day 14 raw cumulative views absent   PASS       
14   day 14 development coverage exact   PASS       
15         day 14 reserved rows absent   PASS       
16  day 14 bundle reloads and predicts   PASS       
17        day 21 raw subscribers saved   PASS 

## Checkpoint decision

Select the subscriber representation using the grouped cross-validation comparison. Raw cumulative channel views remain excluded, and the reserved test remains untouched.